# 03b · Monte Carlo Project Finance Model

This notebook is the **payoff** of the series. It takes the *same* project finance model from notebook `01` and runs it across the **1,000 synthetic weather-and-price futures** generated in `03a` (`generate-synthetic-data.ipynb`) — turning a single deterministic answer into a full **probability distribution of equity returns**.

The logic is unchanged from notebook `01`; the only difference is that two inputs are now **time-varying arrays** rather than constants:

- `net_equivalent_hours` — a per-year production figure, derived from each simulation's stochastic solar resource.
- `merchant_price` — a per-year merchant electricity price, taken from each simulation's stochastic price path.

For each of the 1,000 simulations we rebuild the inputs, solve the full circular model, and record the equity **IRR** and **cash-on-cash** multiple. The histogram of those 1,000 IRRs is the deliverable: instead of "the IRR is X%", we can now say "the IRR is distributed like *this*, with this much downside risk" — exactly the kind of statement an investment committee or lender needs.

## Pipeline in this notebook

1. **Import** the synthetic dataset bundle (`data/PV_SYNTHETIC_DATA/` — a metadata JSON plus one CSV per simulation).
2. **Transform** each simulation's daily CSI/price into the annual model inputs the project finance model expects.
3. **Run** all 1,000 scenarios through the model with the `Scenarios` helper.
4. **Analyse** the resulting distribution of IRR / cash-on-cash.

In [16]:
from finmodel.model import Model, FormulaRow, SimpleRow, PredefinedFormats, PredefinedStyles, row, Format
from finmodel.scenarios import Scenarios
import pandas as pd
from pydantic import BaseModel
from datetime import datetime, timedelta
from dataclasses import dataclass, field, replace, asdict
from dateutil.relativedelta import relativedelta
import numpy as np
import numpy_financial as npf
import time

import plotly.graph_objects as go
import plotly.io as pio
from utils import plotly_setup
from utils.custom_widgets import ProgressBarDark
import json
import io

pio.templates.default = "theme_dark_1"

# Import Synthetic Data

In [2]:
import os

# The synthetic data is a directory bundle (one CSV per simulation) rather than a single
# multi-GB JSON. We read the metadata JSON, which holds the generation parameters and the
# relative path of every simulation file.
synth_dir = r'data\PV_SYNTHETIC_DATA'
synth_metadata_file = os.path.join(synth_dir, 'PV_SYNTHETIC_DATA.json')

with open(synth_metadata_file, 'r') as f:
    metadata = json.load(f)

synth_params = metadata['parameters']
simulation_files = metadata['simulations']

print(f"{len(simulation_files)} simulations | "
      f"{synth_params['years']}y horizon from {synth_params['start_date']} "
      f"@ ({synth_params['lat']}, {synth_params['lon']})")

1000 simulations | 30y horizon from 2027-01-01 @ (38.812821, -6.521058)


The metadata JSON lists one CSV per simulation. We load each CSV, reconstruct actual irradiance as `ghi = clearsky_ghi × csi`, and **resample to annual** figures — summing irradiance (an energy total) but **averaging** price (an intensive quantity). The model runs on annual periods, so this collapses 30 years of daily data into 30 annual rows per simulation.

In [18]:
synth_data = []
import_loop = ProgressBarDark(total=len(simulation_files), desc="Importing data")

for i in import_loop:
    rel_path = simulation_files[i]
    _df = pd.read_csv(os.path.join(synth_dir, rel_path), index_col=0, parse_dates=True)
    _df['ghi'] = _df['clearsky_ghi']*_df['csi']

    agg_dict = {col: 'sum' for col in _df.columns}

    # 3. Manually override the specific column you want as a 'mean'
    # (Replace 'your_mean_column' with the actual name of your column)
    agg_dict['price'] = 'mean'

    # 4. Resample and apply the custom aggregations
    _df = _df.resample("YE").agg(agg_dict)

    synth_data.append(_df.copy())

Importing data:   0%|          | 0/1000 [00:00<?, ?it/s]

### From irradiance to production hours

Annual GHI (energy hitting a flat surface) is converted into **net equivalent full-load hours** — the production input the financial model uses — via two standard PV engineering factors:

- **Transposition factor (1.3)** — the gain from tilting/orienting the panels toward the sun versus a horizontal surface (GHI → plane-of-array irradiance, GTI).
- **Performance ratio (0.8)** — system losses (inverter, temperature, soiling, wiring) between incident irradiance and delivered AC energy.

So `net_eq_hours ≈ (GHI / 1000) × 1.3 × 0.8`. The per-simulation price arrays are carried through unchanged. The two lists (`net_eq_hours`, `price`) hold one length-30 array per simulation.

In [19]:
transposition_factor = 1.3 #Transposition Factor
performance_factor = 0.8 #Performance Ratio

inputs_synth_data = {'net_eq_hours':[], 'price':[]}

for _df in synth_data:
    ghi = _df['ghi']/1e3
    gti = ghi*transposition_factor
    net_eq_hours = gti*performance_factor

    price = _df['price']

    inputs_synth_data['net_eq_hours'].append(net_eq_hours.values)
    inputs_synth_data['price'].append(price.values)

## The project finance model

The `Inputs` dataclass and `PVModel` below are **identical to notebook `01`**, with one change: `net_equivalent_hours` and `merchant_price` are now `np.ndarray` (one value per year) instead of scalars, and the corresponding rows index them by period — `self.inputs.net_equivalent_hours[t]` and `self.inputs.merchant_price[t]`. This lets each year of the model see that simulation's specific resource and price. Everything else — the cash waterfall, DSCR-sculpted debt, the circular interest loop and iterative solver — is exactly as before. The base-case model below is built from the *first* simulation just to confirm it solves.

In [20]:
@dataclass
class Inputs:
    construction_bop: datetime
    construction_periods: relativedelta
    construction_eop: datetime = field(init=False)

    cod_bop: datetime = field(init=False)
    project_life: relativedelta
    cod_eop: datetime = field(init=False)

    installed_capacity: float
    net_equivalent_hours: np.ndarray
    availability: float
    degradation: float
    degradation_start: datetime

    ppa_production: float
    ppa_price: float
    ppa_duration: relativedelta

    merchant_price: np.ndarray

    capex_per_mwp: float
    opex_per_year: float

    days_receivable: float
    days_payable: float

    debt_tenor: relativedelta
    debt_max_leverage_over_capex: float
    debt_dscr: float
    debt_upfront_fee: float
    debt_interest_rate: float

    cpi_bop: datetime
    cpi_per_year: float
    tax_rate: float

    def __post_init__(self):
        self.construction_eop = self.construction_bop + self.construction_periods - timedelta(days=1)
        self.cod_bop = self.construction_eop + timedelta(days=1)
        self.cod_eop = self.cod_bop + self.project_life - timedelta(days=1)

In [21]:
format_3f = Format(formatter=lambda x: f'{x:.3f}')

In [22]:
inputs = Inputs(
    construction_bop=datetime(2023, 1, 1),
    construction_periods=relativedelta(years=1),
    project_life=relativedelta(years=25),
    installed_capacity=10,
    net_equivalent_hours=inputs_synth_data['net_eq_hours'][0],
    availability=0.99,
    degradation=0.5/100,
    degradation_start=datetime(2026, 1, 1),
    ppa_production=0.8,
    ppa_duration=relativedelta(years=13),
    ppa_price=40.0,
    merchant_price=inputs_synth_data['price'][0],
    capex_per_mwp=1000.e3,
    opex_per_year=100.e3,
    days_receivable=30.0,
    days_payable=90.0,
    debt_tenor=relativedelta(years=15),
    debt_max_leverage_over_capex=0.8,
    debt_interest_rate=0.03,
    debt_dscr=1.25,
    debt_upfront_fee=0.02,
    cpi_bop=datetime(2026, 1, 1),
    cpi_per_year=2./100,
    tax_rate=25./100
)

In [23]:
class PVModel(Model):

    period_freq = relativedelta(years=1)

    # -----------------------------------------------
    # PERIODS
    # -----------------------------------------------

    @row(group="Periods", format=PredefinedFormats.DATE)
    def bop(self, t: int):
        if t == 0:
            return self.inputs.construction_bop
        
        return self.bop(t-1) + self.period_freq
    
    @row(group="Periods", format=PredefinedFormats.DATE)
    def eop(self, t: int):
        return self.bop(t) + self.period_freq - relativedelta(days=1)
    
    @row(group="Periods")
    def days(self, t: int):
        return (self.eop(t) - self.bop(t)).days + 1
    

    # -----------------------------------------------
    # FLAGS
    # -----------------------------------------------

    @row(group="Flags")
    def construction(self, t: int):
        return 1*(self.inputs.construction_bop <= self.bop(t) and self.inputs.construction_eop >= self.eop(t))
    
    @row(group="Flags")
    def COD(self, t: int):
        return 1*(self.inputs.cod_bop <= self.bop(t) and self.inputs.cod_eop >= self.eop(t))
    
    @row(group="Flags")
    def years_From_COD(self, t: int):
        if t == 0:
            return 0
        return (self.COD(t) + self.years_From_COD(t-1))*self.COD(t)
    
    @row(group="Flags", format=PredefinedFormats.PERCENTAGE)
    def degradation(self, t: int):
        return self.inputs.degradation
    

    @row(group="Flags", format=PredefinedFormats.PERCENTAGE)
    def degradation_factor(self, t: int):
        if t == 0:
            return 0
        if  self.bop(t) >= self.inputs.degradation_start:
            return self.degradation_factor(t-1)*(1-self.degradation(t))
        else:
            return 1.0
        

    @row(group="Flags")
    def ppa_period(self, t: int):
        return ((self.inputs.cod_bop + self.inputs.ppa_duration) >= self.eop(t))*self.COD(t)
    
    @row(group="Flags")
    def cpi_period(self, t: int):
        return self.bop(t) >= self.inputs.cpi_bop
    
    @row(group="Flags", format=format_3f)
    def cpi_acc(self, t: int):
        if t == 0:
            return 1.0
        return self.cpi_acc(t-1)*(1 + self.inputs.cpi_per_year*self.cpi_period(t))


    # -----------------------------------------------
    # REVENUES
    # -----------------------------------------------


    @row(group="Revenues")
    def installed_capacity(self, t: int):
        return self.inputs.installed_capacity * self.COD(t)
    
    @row(group="Revenues")
    def net_equivalent_hours(self, t: int):
        return self.inputs.net_equivalent_hours[t] * self.COD(t)
    
    @row(group="Revenues", format=PredefinedFormats.PERCENTAGE)
    def availability(self, t: int):
        return self.inputs.availability * self.COD(t)
    
    @row(group="Revenues")
    def production(self, t: int):
        return self.installed_capacity(t)*self.net_equivalent_hours(t)*self.availability(t)*self.degradation_factor(t)
    
    @row(group="Revenues")
    def production_ppa(self, t: int):
        return self.production(t)*self.ppa_period(t)*self.inputs.ppa_production
    
    @row(group="Revenues")
    def production_merchant(self, t: int):
        return self.production(t) - self.production_ppa(t)
    
    @row(group="Revenues")
    def price_ppa(self, t: int):
        return self.inputs.ppa_price * self.COD(t) * self.cpi_acc(t)
    
    @row(group="Revenues")
    def price_merchant(self, t: int):
        return self.inputs.merchant_price[t] * self.COD(t) * self.cpi_acc(t)
    
    @row(group="Revenues")
    def revenues_ppa(self, t: int):
        return self.price_ppa(t) * self.production_ppa(t)/1e3
    
    @row(group="Revenues")
    def revenues_merchant(self, t: int):
        return self.price_merchant(t) * self.production_merchant(t)/1e3
    

    @row(group="Revenues", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def total_revenues(self, t: int):
        return self.revenues_ppa(t) + self.revenues_merchant(t)
    

    # -----------------------------------------------
    # NET FIXED ASSETS
    # -----------------------------------------------
    

    @row(group="Net Fixed Assets")
    def capex(self, t: int):
        construction_periods = sum(self.construction(t_) for t_ in range(self.periods))
        return self.construction(t)*(self.inputs.capex_per_mwp*self.inputs.installed_capacity/1e3)/construction_periods
    

    @row(group="Net Fixed Assets")
    def da(self, t: int):
        total_capex = sum(self.capex(t_) for t_ in range(self.periods))
        total_cod = sum(self.COD(t_) for t_ in range(self.periods))

        return -self.COD(t)*(total_capex/total_cod)
    
    @row(group="Net Fixed Assets")
    def fixed_assets_bop(self, t: int):
        if t == 0:
            return 0
        return self.fixed_assets_eop(t-1)
    
    @row(group="Net Fixed Assets")
    def fixed_assets_eop(self, t: int):
        return self.fixed_assets_bop(t) + self.da(t) + self.capex(t)
    

    # -----------------------------------------------
    # P&L
    # -----------------------------------------------

    @row(group="Revenues", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def revenues(self, t: int):
        return self.total_revenues(t)
    
    @row(group="Revenues")
    def opex(self, t: int):
        return self.COD(t) * (-self.inputs.opex_per_year/1e3)* self.cpi_acc(t)
    
    @row(group="Revenues", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def ebitda(self, t: int):
        return self.revenues(t) + self.opex(t)
    
    @row(group="Revenues", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def ebit(self, t: int):
        return self.ebitda(t) + self.da(t)


    @row(group="Revenues",)
    def net_financial_expenses(self, t: int):
        return self.financial_expenses(t)
    
    @row(group="Revenues", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def ebt(self, t: int):
        return self.ebit(t) + self.net_financial_expenses(t)
    
    @row(group="Revenues")
    def taxes(self, t: int):
        return self.ebt(t) * (-self.inputs.tax_rate)
    
    @row(group="Revenues", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def net_income(self, t: int):
        return self.ebt(t) + self.taxes(t)
    

    # -----------------------------------------------
    # Balance
    # -----------------------------------------------

    @row(group="Balance")
    def net_fixed_assets(self, t: int):
        return self.fixed_assets_eop(t)
    
    @row(group="Balance")
    def receivable(self, t: int):
        return self.revenues(t)*self.inputs.days_receivable/self.days(t) 

    @row(group="Balance")
    def cash(self, t: int):
        return self.cash_eop(t)

    @row(group="Balance", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def total_assets(self, t: int):
        return  self.net_fixed_assets(t) + self.receivable(t) + self.cash(t)
    

    @row(group="Balance")
    def equity(self, t: int):
        return self.equity_eop(t)
    
    @row(group="Balance")
    def payables(self, t: int):
        return -1*self.opex(t)*self.inputs.days_payable/self.days(t)


    @row(group="Balance")
    def debt(self, t: int):
        return self.debt_eop(t)
    
    @row(group="Balance", format=PredefinedFormats.SUMMARY_HIGHLIGHT)
    def total_liabilities(self, t: int):
        return  self.payables(t) + self.equity(t) + self.debt(t)

    
    @row(group="Balance", format=PredefinedFormats.BOOLEAN)
    def balance_check(self, t: int):
        return abs(self.total_assets(t) - self.total_liabilities(t)) < 0.1

    # -----------------------------------------------
    # Cash Flow
    # -----------------------------------------------

    @row(group="Cash Flow")
    def working_capital(self, t: int):
        if t == 0:
            return 0
        return (self.receivable(t-1) - self.payables(t-1)) - (self.receivable(t) - self.payables(t))
    
    @row(group="Cash Flow")
    def cf_to_debt_service(self, t: int):
        return self.ebitda(t) - self.capex(t) + self.working_capital(t) + self.taxes(t)
    
    @row(group="Cash Flow")
    def cf_to_debt_repayment(self, t: int):
        return self.cf_to_debt_service(t) + self.net_financial_expenses(t)
    

    @row(group="Cash Flow")
    def cf_to_shareholders(self, t: int):
        return self.cf_to_debt_repayment(t) + self.debt_repayment(t) + self.debt_drawdown(t)
            

    @row(group="Cash Flow")
    def cash_variation(self, t: int):
        return self.cf_to_shareholders(t) + self.capital_increase(t) + self.dividends(t)

    @row(group="Cash Flow")
    def cash_bop(self, t: int):
        if t == 0:
            return 0
        return self.cash_eop(t-1)
    
    @row(group="Cash Flow")
    def cash_eop(self, t: int):
        return self.cash_bop(t) + self.cash_variation(t)
    
    # -----------------------------------------------
    # Debt
    # -----------------------------------------------

    @row(group="Debt")
    def tenor_debt_flag(self, t: int):
        return self.COD(t)*(t <= self.inputs.debt_tenor.years)
    

    @row()
    def debt_bop(self, t: int):
        if t == 0:
            return 0
        return self.debt_eop(t-1)
    
    @row(group="Debt")
    def debt_drawdown(self, t: int):
        total_debt = sum(self.debt_repayment(t_) for t_ in range(self.periods))
        if t==0:
            return -total_debt
        else:
            return 0
    
    @row()
    def debt_service(self, t: int):
        return self.tenor_debt_flag(t)*self.cf_to_debt_service(t)/self.inputs.debt_dscr

    @row()
    def debt_repayment(self, t: int):
        return -(self.debt_service(t) + self.financial_expenses(t)) * self.tenor_debt_flag(t)
    
    @row()
    def debt_eop(self, t: int):
        return  self.debt_bop(t) + self.debt_repayment(t) + self.debt_drawdown(t)
    

    @row()
    def debt_average(self, t: int):
        return self.debt_bop(t) + self.debt_drawdown(t)/2 + self.debt_repayment(t)*(1/4)

    @row()
    def financial_expenses(self, t: int):
        return self.debt_average(t)*(-self.inputs.debt_interest_rate)


    @row(group="Equity")
    def total_uses(self, t: int):
        return self.capex(t)
    
    @row(group="Equity")
    def total_sources(self, t: int):
        return self.total_uses(t)


    @row(group="Equity")
    def equity_bop(self, t: int):
        if t == 0:
            return 0
        return self.equity_eop(t-1)
    
    
    @row(group="Equity")
    def capital_increase(self, t: int):
        total_debt = sum(self.debt_repayment(t_) for t_ in range(self.periods))
        return (self.total_sources(t) + total_debt)*self.construction(t)
    
    @row(group="Equity")
    def dividends(self, t: int):
        return -max(0, min(self.equity_bop(t), self.cf_to_shareholders(t)) )
    
    @row(group="Equity")
    def net_income_to_equity(self, t: int):
        return self.net_income(t)
    
    @row(group="Equity")
    def equity_eop(self, t: int):
        return self.equity_bop(t) + self.capital_increase(t) + self.net_income_to_equity(t) + self.dividends(t)

    
    @row(group="Ratios", format=PredefinedFormats.PERCENTAGE)
    def debt_over_capex(self, t: int):
        if t == 0:
            total_debt = sum(self.debt_repayment(t_) for t_ in range(self.periods))
            total_capex = sum(self.capex(t_) for t_ in range(self.periods))
            return -total_debt/total_capex
        else:
            return 0



In [24]:
model = PVModel(
    periods=26,
    inputs=inputs,
    style=PredefinedStyles.JETBRAINS_DARK_THEME,
    enable_iterative_calculation=True,
    max_iterations=100,   # was 10; with damping=1.0 the model converges by iter 8-9
    damping=1.0,        # was default 0.5, which over-damps and prevents convergence here
)


model.calculate()
model.show()

## Running the Monte Carlo

This is the core of the analysis. We define the same two equity-return outputs as notebook `01` — **cash-on-cash** and **IRR** — then build one input set per simulation (each with its own `net_equivalent_hours` and `merchant_price` arrays) and solve the **full model 1,000 times**. Because the model is circular, every one of those solves runs the iterative solver to convergence, so this cell is the expensive step (~1 min).

In [26]:
scenarios = Scenarios(model)

def output_coc(model:PVModel):
    dividends = -model.get_result_data('dividends')
    capital_invested =  model.get_result_data('capital_increase')
    return float(np.sum(dividends)/np.sum(capital_invested))
    
def output_irr(model:PVModel):
    dividends = -model.get_result_data('dividends')
    capital_invested =  model.get_result_data('capital_increase')
    returns = dividends - capital_invested
    return float(npf.irr(returns))

scenarios.add_output('coc', output_coc)
scenarios.add_output('irr', output_irr)

scenarios_inputs = []
for i in range(len(inputs_synth_data['net_eq_hours'])):
    scenarios_inputs.append(
        replace(inputs,
                net_equivalent_hours=inputs_synth_data['net_eq_hours'][i],
                merchant_price=inputs_synth_data['price'][i])

    )

scenarios.run_scenarios(scenarios_inputs)
df = scenarios.get_scenarios_df()

Calculating scenarios: 100%|██████████| 1000/1000 [02:30<00:00,  6.63it/s]


In [27]:
df.head()

input                                                       \
  construction_bop     construction_periods construction_eop    cod_bop   
0       2023-01-01  relativedelta(years=+1)       2023-12-31 2024-01-01   
1       2023-01-01  relativedelta(years=+1)       2023-12-31 2024-01-01   
2       2023-01-01  relativedelta(years=+1)       2023-12-31 2024-01-01   
3       2023-01-01  relativedelta(years=+1)       2023-12-31 2024-01-01   
4       2023-01-01  relativedelta(years=+1)       2023-12-31 2024-01-01   

                                                           \
               project_life    cod_eop installed_capacity   
0  relativedelta(years=+25) 2048-12-31                 10   
1  relativedelta(years=+25) 2048-12-31                 10   
2  relativedelta(years=+25) 2048-12-31                 10   
3  relativedelta(years=+25) 2048-12-31                 10   
4  relativedelta(years=+25) 2048-12-31                 10   

                                                                               \
                                net_equivalent_hours availability degradation   
0  [1806.817328186183, 1838.5357534978882, 1868.7...         0.99       0.005   
1  [1733.9044715954435, 1790.3448072917909, 1763....         0.99       0.005   
2  [1815.5999123949387, 1836.2664516881582, 1856....         0.99       0.005   
3  [1838.998864176534, 1781.328383059079, 1766.22...         0.99       0.005   
4  [1743.8450516473754, 1802.3725874219647, 1784....         0.99       0.005   

   ...                                                                   \
   ...                debt_tenor debt_max_leverage_over_capex debt_dscr   
0  ...  relativedelta(years=+15)                          0.8      1.25   
1  ...  relativedelta(years=+15)                          0.8      1.25   
2  ...  relativedelta(years=+15)                          0.8      1.25   
3  ...  relativedelta(years=+15)                          0.8      1.25   
4  ...  relativedelta(years=+15)                          0.8      1.25   

                                                                        \
  debt_upfront_fee debt_interest_rate    cpi_bop cpi_per_year tax_rate   
0             0.02               0.03 2026-01-01         0.02     0.25   
1             0.02               0.03 2026-01-01         0.02     0.25   
2             0.02               0.03 2026-01-01         0.02     0.25   
3             0.02               0.03 2026-01-01         0.02     0.25   
4             0.02               0.03 2026-01-01         0.02     0.25   

     output            
        coc       irr  
0  2.259893  0.049823  
1  2.180082  0.047435  
2  2.325388  0.051697  
3  2.194377  0.047931  
4  2.237569  0.049368  

[5 rows x 29 columns]

## Results — the return distribution

The scenario DataFrame now holds 1,000 rows, one per simulated future. Two views follow:

- **IRR vs. total equivalent hours** — a scatter showing how each path's lifetime resource maps to its return, the probabilistic analogue of notebook `01`'s deterministic sensitivity curve.
- **IRR histogram** — the headline output: the full **probability distribution of equity IRR**. Its centre is the expected return, while its left tail quantifies **downside risk** (e.g. the probability the project underperforms a hurdle rate). This is what the whole pipeline — real data (`02`) → stochastic model (`03a`) → Monte Carlo (`03b`) — was built to produce.

In [28]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x= df[('input', 'net_equivalent_hours')].apply(np.sum),
    y= df[('output', 'irr')],
    mode="markers"
    
))

fig.update_layout(
    title="Returns",
    xaxis=dict(
        title="Net Equivalent Hours",
        tickformat=".0f"
    ),
    yaxis=dict(
        title="IRR",
        tickformat=".1%"
    )
)

fig.show()

In [29]:
fig = go.Figure()

# 1. Add Trace 1: Revenues (Mapped to primary 'x')
fig.add_trace(
    go.Histogram(
        x=df[('output', 'irr')],
        opacity=0.8,
        histnorm='probability'
    )
)

fig.update_layout(
    title="IRR",
    xaxis=dict(
        tickformat='.1%'
    ),
    yaxis=dict(
        tickformat='.1%'
    )
)


fig.show()